In [3]:
# 🏀 March Madness 2025 - Enhanced Prediction Model
# Combining solid fundamentals from Notebook 1 with advanced features from Notebook 2

# 🏀 March Madness 2025 - Enhanced Prediction Model

import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
import seaborn as sns

print("🏀 March Madness 2025 Prediction Model 🏀")
start_time = time.time()

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
print("Loading data files...")
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))

mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))

mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

mteam_conferences = pd.read_csv(os.path.join(data_path, "MTeamConferences.csv"))
wteam_conferences = pd.read_csv(os.path.join(data_path, "WTeamConferences.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed_Numeric"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed_Numeric"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics
def compute_team_stats(results, conferences):
    print("Processing team statistics...")
    
    # Winning Stats
    win_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "std", "count"],
        "LScore": ["mean", "std"],
        "NumOT": "sum",
        "WFGM": "sum", "WFGA": "sum",
        "WFGM3": "sum", "WFGA3": "sum",
        "WFTM": "sum", "WFTA": "sum",
        "WOR": "sum", "WDR": "sum",
        "WAst": "sum", "WTO": "sum",
        "WStl": "sum", "WBlk": "sum",
        "WPF": "sum",
    }).reset_index()

    # Rename columns properly
    win_stats.columns = ["_".join(col).replace(" ", "_") if isinstance(col, tuple) else col for col in win_stats.columns]
    win_stats.rename(columns={"WTeamID": "TeamID"}, inplace=True)

    # Losing Stats
    loss_stats = results.groupby(["Season", "LTeamID"]).agg({
        "LScore": ["mean", "std", "count"],
        "WScore": ["mean", "std"],
        "NumOT": "sum",
        "LFGM": "sum", "LFGA": "sum",
        "LFGM3": "sum", "LFGA3": "sum",
        "LFTM": "sum", "LFTA": "sum",
        "LOR": "sum", "LDR": "sum",
        "LAst": "sum", "LTO": "sum",
        "LStl": "sum", "LBlk": "sum",
        "LPF": "sum",
    }).reset_index()

    # Rename columns properly
    loss_stats.columns = ["_".join(col).replace(" ", "_") if isinstance(col, tuple) else col for col in loss_stats.columns]
    loss_stats.rename(columns={"LTeamID": "TeamID"}, inplace=True)

    # Merge Win & Loss Stats
    team_stats = win_stats.merge(loss_stats, on=["Season", "TeamID"], how="outer", suffixes=("_Win", "_Loss"))
    team_stats.fillna(0, inplace=True)

    # Merge conference information
    team_stats = team_stats.merge(conferences, on=["Season", "TeamID"], how="left")

    # Feature Engineering
    team_stats["WinPct"] = team_stats["WScore_count"] / (team_stats["WScore_count"] + team_stats["LScore_count"])
    team_stats["PointMargin"] = team_stats["WScore_mean"] - team_stats["LScore_mean"]
    team_stats["FG_Pct"] = team_stats["WFGM_sum"] / team_stats["WFGA_sum"]
    team_stats["ThreePt_Pct"] = team_stats["WFGM3_sum"] / team_stats["WFGA3_sum"]
    team_stats["FT_Pct"] = team_stats["WFTM_sum"] / team_stats["WFTA_sum"]
    
    # Defensive & Rebounding Metrics
    team_stats["OffReboundPct"] = team_stats["WOR_sum"] / (team_stats["WOR_sum"] + team_stats["LDR_sum"])
    team_stats["DefReboundPct"] = team_stats["WDR_sum"] / (team_stats["WDR_sum"] + team_stats["LOR_sum"])
    
    return team_stats

print("Computing team statistics...")
mteam_stats = compute_team_stats(mregular_results, mteam_conferences)
wteam_stats = compute_team_stats(wregular_results, wteam_conferences)

# ✅ Training Data Preparation
def prepare_training_data(results, seeds, team_stats):
    features = results.copy()
    features = features.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed_Numeric": "WSeed"}).drop(columns=["TeamID"])
    features = features.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed_Numeric": "LSeed"}).drop(columns=["TeamID"])
    features = features.merge(team_stats, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").drop(columns=["TeamID"]).rename(columns={col: f"W_{col}" for col in team_stats.columns if col not in ["Season", "TeamID"]})
    features = features.merge(team_stats, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").drop(columns=["TeamID"]).rename(columns={col: f"L_{col}" for col in team_stats.columns if col not in ["Season", "TeamID"]})
    features.fillna(0, inplace=True)

    features["SeedDiff"] = features["WSeed"] - features["LSeed"]
    features["PointMarginDiff"] = features["W_PointMargin"] - features["L_PointMargin"]
    features["WinPctDiff"] = features["W_WinPct"] - features["L_WinPct"]

    # Balanced dataset with both win/loss scenarios
    team_a_wins = features[["SeedDiff", "PointMarginDiff", "WinPctDiff"]].copy()
    team_a_wins["Result"] = 1
    team_b_wins = team_a_wins.copy()
    team_b_wins.iloc[:, :-1] *= -1
    team_b_wins["Result"] = 0

    return pd.concat([team_a_wins, team_b_wins], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# ✅ Save Preprocessed Data
full_train_data.to_csv("processed_training_data.csv", index=False)

print("✅ Data processing complete!")

# ✅ Feature importance analysis (will be used later)
def plot_feature_importance(model, X_columns):
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        # Sort importance values
        indices = np.argsort(importance)[::-1]
        
        plt.figure(figsize=(10, 8))
        plt.title('Feature Importances')
        plt.barh(range(len(indices)), importance[indices], align='center')
        plt.yticks(range(len(indices)), [X_columns[i] for i in indices])
        plt.xlabel('Relative Importance')
        plt.tight_layout()
        plt.show()

# ✅ Prepare Training Data with a balanced approach from both notebooks
def prepare_training_data(results, seeds, team_stats):
    # Start with tournament results
    features = results.copy()
    
    # Merge seed information for both winning and losing teams
    features = features.merge(
        seeds[["Season", "TeamID", "Seed_Numeric"]], 
        left_on=["Season", "WTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    ).rename(columns={"Seed_Numeric": "WSeed"}).drop(columns=["TeamID"])
    
    features = features.merge(
        seeds[["Season", "TeamID", "Seed_Numeric"]], 
        left_on=["Season", "LTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    ).rename(columns={"Seed_Numeric": "LSeed"}).drop(columns=["TeamID"])
    
    # Merge team statistics for both teams
    features = features.merge(
        team_stats, 
        left_on=["Season", "WTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    ).drop(columns=["TeamID"]).rename(columns={
        col: f"W_{col}" for col in team_stats.columns if col not in ["Season", "TeamID"]
    })
    
    features = features.merge(
        team_stats, 
        left_on=["Season", "LTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    ).drop(columns=["TeamID"]).rename(columns={
        col: f"L_{col}" for col in team_stats.columns if col not in ["Season", "TeamID"]
    })
    
    # Fill NaN values that might have been introduced
    features.fillna(0, inplace=True)
    
    # Create feature differences (Team A vs Team B metrics)
    features["SeedDiff"] = features["WSeed"] - features["LSeed"]
    features["WinPctDiff"] = features["W_WinPct"] - features["L_WinPct"]
    features["PointMarginDiff"] = features["W_PointMargin_Avg"] - features["L_PointMargin_Avg"]
    features["FGPctDiff"] = features["W_FG_Pct"] - features["L_FG_Pct"]
    features["FG3PctDiff"] = features["W_FG3_Pct"] - features["L_FG3_Pct"]
    features["FTPctDiff"] = features["W_FT_Pct"] - features["L_FT_Pct"]
    features["ReboundMarginDiff"] = features["W_ReboundMargin"] - features["L_ReboundMargin"]
    features["AssistTORatioDiff"] = features["W_AssistTurnoverRatio"] - features["L_AssistTurnoverRatio"]
    features["BlocksPerGameDiff"] = features["W_BlocksPerGame"] - features["L_BlocksPerGame"]
    features["StealsPerGameDiff"] = features["W_StealsPerGame"] - features["L_StealsPerGame"]
    features["DefensiveRatingDiff"] = features["W_DefensiveRating"] - features["L_DefensiveRating"]
    features["ScoringConsistencyDiff"] = features["W_ScoringConsistency"] - features["L_ScoringConsistency"]
    
    # Select which features to use for modeling
    feature_columns = [
        "SeedDiff", "WinPctDiff", "PointMarginDiff", "FGPctDiff", "FG3PctDiff", 
        "FTPctDiff", "ReboundMarginDiff", "AssistTORatioDiff", "BlocksPerGameDiff", 
        "StealsPerGameDiff", "DefensiveRatingDiff", "ScoringConsistencyDiff"
    ]
    
    # Create balanced dataset with both win and loss scenarios
    # Original matchups (Team A wins)
    team_a_wins = features[["Season", "WTeamID", "LTeamID"] + feature_columns].copy()
    team_a_wins["Result"] = 1  # Team A wins
    
    # Now create the opposite scenario by swapping teams and reversing all difference metrics
    team_b_wins = features[["Season", "LTeamID", "WTeamID"] + feature_columns].copy()
    team_b_wins.columns = team_a_wins.columns  # Ensure same column names after the swap
    
    # Reverse the sign of all difference features
    for col in feature_columns:
        team_b_wins[col] = -team_b_wins[col]
    
    team_b_wins["Result"] = 0  # Team A loses (Team B wins)
    
    # Combine both scenarios
    balanced_features = pd.concat([team_a_wins, team_b_wins], ignore_index=True)
    
    return balanced_features

print("Preparing training data...")
mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)

# Combine men's and women's data for a larger training set
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# Look at class balance
print(f"Class distribution:\n{full_train_data['Result'].value_counts(normalize=True) * 100}")

# ✅ Split and Scale Data
X = full_train_data.drop(columns=["Season", "WTeamID", "LTeamID", "Result"])
y = full_train_data["Result"]

# Keep track of feature names
feature_names = X.columns

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Validation set size: {X_valid.shape}")

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# ✅ Train and evaluate multiple models (from Notebook 1)
def evaluate_model(model, X_train, y_train, X_valid, y_valid, name):
    start = time.time()
    model.fit(X_train, y_train)
    
    # Make predictions on validation set
    y_pred_proba = model.predict_proba(X_valid)[:,1]
    score = log_loss(y_valid, y_pred_proba)
    
    # Get cross-validation score
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, scoring="neg_log_loss", cv=cv)
    cv_score = -np.mean(cv_scores)
    
    end = time.time()
    print(f"{name} - Validation Log Loss: {score:.4f}, CV Log Loss: {cv_score:.4f}, Time: {end-start:.2f} seconds")
    
    return model, score, cv_score

print("\nTraining and evaluating models...")
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.1),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42)
}

trained_models = {}
best_model = None
best_score = float("inf")

for name, model in models.items():
    trained_model, score, cv_score = evaluate_model(model, X_train_scaled, y_train, X_valid_scaled, y_valid, name)
    trained_models[name] = (trained_model, score, cv_score)
    
    if score < best_score:
        best_score = score
        best_model = trained_model
        best_model_name = name

print(f"\nBest model: {best_model_name} with validation log loss: {best_score:.4f}")

# ✅ Create a stacked ensemble model (from Notebook 2)
print("\nTraining stacked ensemble model...")
base_models = [
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
    ("xgb", XGBClassifier(n_estimators=100, learning_rate=0.05, random_state=42))
]

stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5
)

stacked_model, stacked_score, stacked_cv_score = evaluate_model(
    stacked_model, X_train_scaled, y_train, X_valid_scaled, y_valid, "Stacked Ensemble"
)

# Update best model if stacked model is better
if stacked_score < best_score:
    best_score = stacked_score
    best_model = stacked_model
    best_model_name = "Stacked Ensemble"
    print(f"Stacked Ensemble is now the best model with log loss: {best_score:.4f}")

# ✅ Plot feature importance for the best model if it's a tree-based model
if best_model_name in ["Random Forest", "Gradient Boosting", "XGBoost"]:
    print("\nPlotting feature importance...")
    plot_feature_importance(best_model, feature_names)

# ✅ Generate predictions for the test dataset
print("\nGenerating tournament predictions...")
submission_df = pd.read_csv(os.path.join(data_path, "SampleSubmissionStage2.csv"))

# Extract Season, WTeamID, and LTeamID from the ID
submission_df[["Season", "WTeamID", "LTeamID"]] = submission_df["ID"].str.split("_", expand=True).astype(int)

# Prepare features for prediction
submission_features = []

for _, row in submission_df.iterrows():
    season = row["Season"]
    team_a = row["WTeamID"]
    team_b = row["LTeamID"]
    
    # Get seed information for both teams
    team_a_seed = mtourney_seeds[(mtourney_seeds["Season"] == season) & (mtourney_seeds["TeamID"] == team_a)]["Seed_Numeric"].values
    team_a_seed = team_a_seed[0] if len(team_a_seed) > 0 else 0
    
    team_b_seed = mtourney_seeds[(mtourney_seeds["Season"] == season) & (mtourney_seeds["TeamID"] == team_b)]["Seed_Numeric"].values
    team_b_seed = team_b_seed[0] if len(team_b_seed) > 0 else 0
    
    # Get team stats for both teams
    team_a_stats = mteam_stats[(mteam_stats["Season"] == season) & (mteam_stats["TeamID"] == team_a)]
    team_b_stats = mteam_stats[(mteam_stats["Season"] == season) & (mteam_stats["TeamID"] == team_b)]
    
    # Skip if stats not found for either team
    if team_a_stats.empty or team_b_stats.empty:
        submission_features.append({
            "SeedDiff": team_a_seed - team_b_seed,
            "WinPctDiff": 0,
            "PointMarginDiff": 0,
            "FGPctDiff": 0,
            "FG3PctDiff": 0,
            "FTPctDiff": 0,
            "ReboundMarginDiff": 0,
            "AssistTORatioDiff": 0,
            "BlocksPerGameDiff": 0,
            "StealsPerGameDiff": 0,
            "DefensiveRatingDiff": 0,
            "ScoringConsistencyDiff": 0
        })
        continue
    
    # Calculate feature differences
    feature_diff = {
        "SeedDiff": team_a_seed - team_b_seed,
        "WinPctDiff": team_a_stats["WinPct"].values[0] - team_b_stats["WinPct"].values[0],
        "PointMarginDiff": team_a_stats["PointMargin_Avg"].values[0] - team_b_stats["PointMargin_Avg"].values[0],
        "FGPctDiff": team_a_stats["FG_Pct"].values[0] - team_b_stats["FG_Pct"].values[0],
        "FG3PctDiff": team_a_stats["FG3_Pct"].values[0] - team_b_stats["FG3_Pct"].values[0],
        "FTPctDiff": team_a_stats["FT_Pct"].values[0] - team_b_stats["FT_Pct"].values[0],
        "ReboundMarginDiff": team_a_stats["ReboundMargin"].values[0] - team_b_stats["ReboundMargin"].values[0],
        "AssistTORatioDiff": team_a_stats["AssistTurnoverRatio"].values[0] - team_b_stats["AssistTurnoverRatio"].values[0],
        "BlocksPerGameDiff": team_a_stats["BlocksPerGame"].values[0] - team_b_stats["BlocksPerGame"].values[0],
        "StealsPerGameDiff": team_a_stats["StealsPerGame"].values[0] - team_b_stats["StealsPerGame"].values[0],
        "DefensiveRatingDiff": team_a_stats["DefensiveRating"].values[0] - team_b_stats["DefensiveRating"].values[0],
        "ScoringConsistencyDiff": team_a_stats["ScoringConsistency"].values[0] - team_b_stats["ScoringConsistency"].values[0]
    }
    
    submission_features.append(feature_diff)

# Convert to DataFrame
submission_features_df = pd.DataFrame(submission_features)

# Scale features
submission_features_scaled = scaler.transform(submission_features_df)

# Generate predictions
submission_df["Pred"] = best_model.predict_proba(submission_features_scaled)[:, 1]

# Save submission file
submission_df[["ID", "Pred"]].to_csv("march_madness_2025_predictions.csv", index=False)

print("\n✅ Submission file generated successfully!")
print(f"Total execution time: {(time.time() - start_time)/60:.2f} minutes")

🏀 March Madness 2025 Prediction Model 🏀
Loading data files...
Computing team statistics...
Processing team statistics...


KeyError: 'Season'

In [16]:
# 🏀 March Madness 2025 - Enhanced Prediction Model
# Combines fundamental features with advanced analytics for robust predictions

import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
import seaborn as sns

print("🏀 March Madness 2025 Prediction Model 🏀")
start_time = time.time()

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
print("Loading data files...")
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))

mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))

mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

mteam_conferences = pd.read_csv(os.path.join(data_path, "MTeamConferences.csv"))
wteam_conferences = pd.read_csv(os.path.join(data_path, "WTeamConferences.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed_Numeric"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed_Numeric"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics with Explicit Season Column
def compute_team_stats(results, conferences):
    print("Processing team statistics...")

    # ✅ Ensure 'Season' column exists before grouping
    if "Season" not in results.columns:
        raise KeyError("❌ The 'Season' column is missing from the input results!")

    # Winning team statistics
    win_stats = results.groupby(["Season", "WTeamID"], as_index=False).agg({
        "WScore": ["mean", "std", "count"],
        "LScore": ["mean", "std"],
        "NumOT": "sum",
        "WFGM": "sum", "WFGA": "sum",
        "WFGM3": "sum", "WFGA3": "sum",
        "WFTM": "sum", "WFTA": "sum",
        "WOR": "sum", "WDR": "sum",
        "WAst": "sum", "WTO": "sum",
        "WStl": "sum", "WBlk": "sum",
        "WPF": "sum",
    }).reset_index()

    # ✅ Ensure 'Season' is retained
    if "Season" not in win_stats.columns:
        raise KeyError("❌ 'Season' was dropped in win_stats!")

    # Flatten column names
    win_stats.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in win_stats.columns]
    win_stats.rename(columns={"WTeamID": "TeamID"}, inplace=True)

    # Losing team statistics
    loss_stats = results.groupby(["Season", "LTeamID"], as_index=False).agg({
        "LScore": ["mean", "std", "count"],
        "WScore": ["mean", "std"],
        "NumOT": "sum",
        "LFGM": "sum", "LFGA": "sum",
        "LFGM3": "sum", "LFGA3": "sum",
        "LFTM": "sum", "LFTA": "sum",
        "LOR": "sum", "LDR": "sum",
        "LAst": "sum", "LTO": "sum",
        "LStl": "sum", "LBlk": "sum",
        "LPF": "sum",
    }).reset_index()

    # ✅ Ensure 'Season' is retained
    if "Season" not in loss_stats.columns:
        raise KeyError("❌ 'Season' was dropped in loss_stats!")

    loss_stats.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in loss_stats.columns]
    loss_stats.rename(columns={"LTeamID": "TeamID"}, inplace=True)

    # ✅ Debugging: Print column names to ensure 'Season' is present
    print("Win Stats Columns:", win_stats.columns)
    print("Loss Stats Columns:", loss_stats.columns)

    # ✅ Ensure 'Season' exists in both DataFrames before merging
    if "Season" not in win_stats.columns or "Season" not in loss_stats.columns:
        raise KeyError("❌ The 'Season' column is missing after aggregation!")

    # Merge statistics
    team_stats = win_stats.merge(loss_stats, on=["Season", "TeamID"], how="outer", suffixes=("_Win", "_Loss"))
    team_stats.fillna(0, inplace=True)

    # Merge conference data
    team_stats = team_stats.merge(conferences, on=["Season", "TeamID"], how="left")

    # ✅ Debugging: Print merged team_stats column names
    print("Final Team Stats Columns:", team_stats.columns)

    # ✅ Updated Feature Calculation (Fixing column names)
    team_stats["WinPct"] = team_stats["WScore_count"] / (team_stats["WScore_count"] + team_stats["LScore_count"] + 1)
    team_stats["PointMargin"] = team_stats["WScore_mean_Win"] - team_stats["LScore_mean_Loss"]
    team_stats["FG_Pct"] = team_stats["WFGM_sum"] / (team_stats["WFGA_sum"] + 1)  # Avoid division by zero

    return team_stats


print("Computing team statistics...")
mteam_stats = compute_team_stats(mregular_results, mteam_conferences)
wteam_stats = compute_team_stats(wregular_results, wteam_conferences)

# ✅ Prepare Training Data
def prepare_training_data(results, seeds, team_stats):
    print("\n🔎 Checking dataset before merging...")
    print("Tournament Results Columns:", results.columns.tolist())

    features = results.copy()

    # Merge seeds
    features = features.merge(
        seeds[["Season", "TeamID", "Seed_Numeric"]],
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left"
    ).rename(columns={"Seed_Numeric": "WSeed"}).drop(columns=["TeamID"])

    features = features.merge(
        seeds[["Season", "TeamID", "Seed_Numeric"]],
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left"
    ).rename(columns={"Seed_Numeric": "LSeed"}).drop(columns=["TeamID"])

    print("\n🔎 Checking columns after merging seeds:", features.columns.tolist())

    # Debug: Print team_stats columns
    print("\n🔎 Checking team_stats columns before merging:", team_stats.columns.tolist())

    # 🚨 Preserve `LTeamID` when merging WTeam stats
    features = features.merge(
        team_stats,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left"
    ).rename(columns=lambda x: f"W_{x}" if x not in ["Season", "WTeamID", "LTeamID"] else x)

    print("\n✅ Merged WTeam stats. Checking columns:", features.columns.tolist())

    if "LTeamID" not in features.columns:
        raise KeyError("❌ 'LTeamID' was dropped after merging WTeam stats!")

    # 🚨 Merge LTeam stats properly
    features = features.merge(
        team_stats,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left"
    ).rename(columns=lambda x: f"L_{x}" if x not in ["Season", "WTeamID", "LTeamID"] else x)

    print("\n✅ Merged LTeam stats. Final columns:", features.columns.tolist())

    if "LTeamID" not in features.columns:
        raise KeyError("❌ 'LTeamID' was dropped after merging LTeam stats!")

    # 🚀 **Add the Result column**
    features["Result"] = 1  # Team A wins
    flipped_features = features.copy()
    
    # Flip teams to create negative examples (Team A loses)
    flipped_features["WTeamID"], flipped_features["LTeamID"] = (
        flipped_features["LTeamID"], flipped_features["WTeamID"]
    )
    
    # Flip all difference-based features
    for col in features.columns:
        if col.startswith("W_"):
            flipped_features[col], flipped_features[col.replace("W_", "L_")] = (
                flipped_features[col.replace("W_", "L_")],
                flipped_features[col],
            )

    flipped_features["Result"] = 0  # Team A loses

    # Merge original and flipped versions to balance training data
    balanced_data = pd.concat([features, flipped_features], ignore_index=True)

    print("\n✅ Training dataset prepared successfully! Checking final columns:", balanced_data.columns.tolist())

    return balanced_data






print("Preparing training data...")
mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

X = full_train_data.drop(columns=["Result"])
y = full_train_data["Result"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Train & Evaluate Models
print("\nTraining and evaluating models...")
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.1),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    score = log_loss(y_valid, model.predict_proba(X_valid_scaled)[:, 1])
    print(f"{name} - Log Loss: {score:.4f}")

print("✅ Model training complete!")
print(f"Total execution time: {(time.time() - start_time)/60:.2f} minutes")


🏀 March Madness 2025 Prediction Model 🏀
Loading data files...
Computing team statistics...
Processing team statistics...
Win Stats Columns: Index(['index', 'Season', 'TeamID', 'WScore_mean', 'WScore_std',
       'WScore_count', 'LScore_mean', 'LScore_std', 'NumOT_sum', 'WFGM_sum',
       'WFGA_sum', 'WFGM3_sum', 'WFGA3_sum', 'WFTM_sum', 'WFTA_sum', 'WOR_sum',
       'WDR_sum', 'WAst_sum', 'WTO_sum', 'WStl_sum', 'WBlk_sum', 'WPF_sum'],
      dtype='object')
Loss Stats Columns: Index(['index', 'Season', 'TeamID', 'LScore_mean', 'LScore_std',
       'LScore_count', 'WScore_mean', 'WScore_std', 'NumOT_sum', 'LFGM_sum',
       'LFGA_sum', 'LFGM3_sum', 'LFGA3_sum', 'LFTM_sum', 'LFTA_sum', 'LOR_sum',
       'LDR_sum', 'LAst_sum', 'LTO_sum', 'LStl_sum', 'LBlk_sum', 'LPF_sum'],
      dtype='object')
Final Team Stats Columns: Index(['index_Win', 'Season', 'TeamID', 'WScore_mean_Win', 'WScore_std_Win',
       'WScore_count', 'LScore_mean_Win', 'LScore_std_Win', 'NumOT_sum_Win',
       'WFGM_sum',

ValueError: could not convert string to float: 'A'

In [ ]:
# 🏀 March Madness 2025 - Optimized Model with Smoothed Elo and Feature Pruning
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import shap

# 📂 Define Data Path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Smoothed Elo Rating Calculation
def compute_smooth_elo(results):
    K = 20  # Lower K-factor for stability
    initial_rating = 1400
    team_ratings = {}

    for team in results["WTeamID"].unique():
        team_ratings[team] = initial_rating

    for index, row in results.iterrows():
        w_team, l_team = row["WTeamID"], row["LTeamID"]

        if w_team not in team_ratings:
            team_ratings[w_team] = initial_rating
        if l_team not in team_ratings:
            team_ratings[l_team] = initial_rating

        # Smooth Elo update with moving average
        exp_w = 1 / (1 + 10 ** ((team_ratings[l_team] - team_ratings[w_team]) / 400))
        team_ratings[w_team] = 0.9 * team_ratings[w_team] + 0.1 * (team_ratings[w_team] + K * (1 - exp_w))
        team_ratings[l_team] = 0.9 * team_ratings[l_team] + 0.1 * (team_ratings[l_team] + K * (0 - exp_w))

    return team_ratings

# Compute Smoothed Elo Ratings
melo_ratings = compute_smooth_elo(mregular_results)
welo_ratings = compute_smooth_elo(wregular_results)

# ✅ Compute Team Statistics
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean"],
        "LScore": ["mean"],
        "WFGM": ["mean"],
        "WFGA": ["mean"],
        "WTO": ["mean"],
        "WStl": ["mean"],
        "WBlk": ["mean"],
    }).reset_index()

    team_stats.columns = ["Season", "TeamID", "AvgWScore", "AvgLScore", "WFGM", "WFGA", "Turnovers", "Steals", "Blocks"]
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats, elo_ratings):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # Merge Smoothed Elo Ratings
    results["WElo"] = results["WTeamID"].map(elo_ratings)
    results["LElo"] = results["LTeamID"].map(elo_ratings)

    # Fill NaN values
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering (Pruning weak features)
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["EloDiff"] = results["WElo"] - results["LElo"]
    results["FGMDiff"] = results["WFGM_W"] - results["WFGM_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]

    # Use SHAP to determine most important features
    feature_cols = ["SeedDiff", "EloDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats, melo_ratings)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats, welo_ratings)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Train XGBoost with New Optimized Settings
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(
    learning_rate=0.05,  # Slightly higher for faster convergence
    n_estimators=600,
    max_depth=5,
    subsample=0.8,  # Introduce some randomness for generalization
    colsample_bytree=0.8
)
xgb.fit(X_train, y_train)

y_pred = xgb.predict_proba(X_valid)[:, 1]
score = log_loss(y_valid, y_pred)

cv_scores = cross_val_score(xgb, X_train, y_train, scoring="neg_log_loss", cv=5)

print(f"Optimized XGBoost Log Loss: {score}")
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")


Optimized XGBoost Log Loss: 0.9374031227697056
Cross-Validation Log Loss: 0.9331038368895245


: 